In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Netflix Recommendation System") \
    .getOrCreate()

print("Spark Ready")

Spark Ready


In [2]:
movies = spark.read.csv(
    r"D:\GitHub\DPySpark\data\movies.csv",
    header=True,
    inferSchema=True
)

ratings = spark.read.csv(
    r"D:\GitHub\DPySpark\data\ratings.csv",
    header=True,
    inferSchema=True
)

In [3]:
movies.show(5)

ratings.show(5)

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows
+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|    296|   5.0|1147880044|
|     1|    306|   3.5|1147868817|
|     1|    307|   5.0|1147868828|
|     1|    665|   5.0|1147878820|
|     1|    899|   3.5|1147868510|
+------+-------+------+----------+
only showing top 5 rows


In [4]:
movies.printSchema()

ratings.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)



In [5]:
print("Movies:", movies.count())

print("Ratings:", ratings.count())

Movies: 62423
Ratings: 25000095


In [6]:
ratings.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)



In [7]:
ratings.count()

25000095

In [8]:
ratings.show(5)

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|    296|   5.0|1147880044|
|     1|    306|   3.5|1147868817|
|     1|    307|   5.0|1147868828|
|     1|    665|   5.0|1147878820|
|     1|    899|   3.5|1147868510|
+------+-------+------+----------+
only showing top 5 rows


In [9]:
spark.range(10).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [10]:
ratings_small = ratings.limit(500000)

ratings_small.count()

500000

In [11]:
from pyspark.ml.recommendation import ALS

In [12]:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=10,
    maxIter=5,
    regParam=0.1,
    coldStartStrategy="drop"
)

In [13]:
model = als.fit(ratings_small)

print("Model Trained")

Model Trained


In [14]:
recommendations = model.recommendForAllUsers(5)

In [15]:
recommendations.show(10, truncate=False)

+------+--------------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                         |
+------+--------------------------------------------------------------------------------------------------------+
|1     |[{114042, 5.3461146}, {108566, 5.302604}, {166291, 5.154707}, {3881, 5.151499}, {65709, 5.058246}]      |
|12    |[{3881, 5.279851}, {69699, 5.1547995}, {65709, 5.0859337}, {117364, 5.0793214}, {60103, 5.0540733}]     |
|22    |[{117364, 6.5502377}, {6653, 6.439192}, {48045, 6.4009585}, {60103, 6.3239408}, {99642, 6.3049245}]     |
|26    |[{117364, 5.437994}, {3881, 5.232638}, {127019, 5.0633163}, {27112, 5.024751}, {92904, 4.9715176}]      |
|27    |[{72039, 6.057493}, {117364, 5.8056917}, {60103, 5.725975}, {189475, 5.6953506}, {107700, 5.6953506}]   |
|28    |[{3881, 6.676594}, {117364, 6.6381273}, {69699, 6.6264944}, {95973, 6.5327444}, 